# 03 Silver - Healthcare

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Casts, normalizes, deduplicates, validates relationships, and quarantines invalid rows.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'healthcare':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp_lab":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from functools import reduce
from pyspark.sql import Window, functions as F

industry = 'healthcare'
specs = {'patients': {'filename': 'patients.csv', 'primary_key': ['patient_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'age_band', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'coverage_type', 'type': 'STRING', 'required': True}, {'name': 'risk_band', 'type': 'STRING', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'providers': {'filename': 'providers.csv', 'primary_key': ['provider_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'specialty', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'facility_type', 'type': 'STRING', 'required': True}, {'name': 'daily_capacity', 'type': 'BIGINT', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'appointments': {'filename': 'appointments.csv', 'primary_key': ['appointment_id'], 'foreign_keys': [['patient_id', 'patients', 'patient_id'], ['provider_id', 'providers', 'provider_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'appointment_id', 'type': 'STRING', 'required': True}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'booked_at', 'type': 'TIMESTAMP', 'required': True}, {'name': 'scheduled_start', 'type': 'TIMESTAMP', 'required': True}, {'name': 'scheduled_end', 'type': 'TIMESTAMP', 'required': True}, {'name': 'appointment_type', 'type': 'STRING', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'encounters': {'filename': 'encounters.csv', 'primary_key': ['encounter_id'], 'foreign_keys': [['appointment_id', 'appointments', 'appointment_id'], ['patient_id', 'patients', 'patient_id'], ['provider_id', 'providers', 'provider_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'encounter_id', 'type': 'STRING', 'required': True}, {'name': 'appointment_id', 'type': 'STRING', 'required': False}, {'name': 'patient_id', 'type': 'STRING', 'required': True}, {'name': 'provider_id', 'type': 'STRING', 'required': True}, {'name': 'encounter_start', 'type': 'TIMESTAMP', 'required': True}, {'name': 'encounter_end', 'type': 'TIMESTAMP', 'required': True}, {'name': 'diagnosis_group', 'type': 'STRING', 'required': True}, {'name': 'procedure_group', 'type': 'STRING', 'required': True}, {'name': 'cost_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'disposition', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"appointments": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/appointments/", "encounters": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/encounters/", "patients": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/patients/", "providers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/healthcare/providers/"}
destinations = {"appointments": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/appointments/", "encounters": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/encounters/", "patients": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/patients/", "providers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/providers/"}
quality_uri = f'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/healthcare/quality_issues/'
spark_types = {"STRING": "string", "DATE": "date", "TIMESTAMP": "timestamp", "DOUBLE": "double", "BIGINT": "bigint", "BOOLEAN": "boolean"}
enum_rules = {"appointments": {"appointment_type": ["consultation", "follow_up", "preventive"], "status": ["completed", "no_show", "scheduled"]}, "encounters": {"diagnosis_group": ["respiratory", "cardiovascular", "musculoskeletal", "preventive"], "disposition": ["outpatient"], "procedure_group": ["evaluation", "imaging", "therapy", "screening"]}, "patients": {"age_band": ["0-17", "18-39", "40-64", "65+"], "coverage_type": ["public", "private", "self_pay"], "region": ["north", "south", "east", "west"], "risk_band": ["low", "medium", "high"], "status": ["active"]}, "providers": {"facility_type": ["clinic", "hospital", "virtual"], "region": ["north", "south", "east", "west"], "specialty": ["primary_care", "cardiology", "orthopedics", "pediatrics"], "status": ["active"]}}
positive_rules = {"encounters": ["cost_amount"], "providers": ["daily_capacity"]}
temporal_rules = {"appointments": [["scheduled_start", "scheduled_end"]], "encounters": [["encounter_start", "encounter_end"]]}

typed = {}
for dataset, spec in specs.items():
    frame = spark.table(table("bronze", dataset))
    cast_failure_columns = []
    for column in spec["columns"]:
        name, kind = column["name"], column["type"]
        raw_text = F.trim(F.col(name).cast("string"))
        if kind == "STRING":
            normalized = F.when(raw_text == "", F.lit(None)).otherwise(raw_text)
            if name not in {"participant_key", "source_row_id"} and not name.endswith("_id"):
                normalized = F.lower(normalized)
            frame = frame.withColumn(name, normalized)
        else:
            flag = f"_invalid_cast_{name}"
            frame = frame.withColumn(
                flag,
                raw_text.isNotNull() & (raw_text != "") & raw_text.cast(spark_types[kind]).isNull(),
            ).withColumn(name, raw_text.cast(spark_types[kind]))
            cast_failure_columns.append(flag)
    cast_invalid = reduce(
        lambda left, name: left | F.col(name), cast_failure_columns, F.lit(False)
    )
    frame = frame.withColumn("_cast_invalid", cast_invalid).drop(*cast_failure_columns)
    typed[dataset] = frame

quality_frames = []
accepted = {}
for dataset, spec in specs.items():
    frame = typed[dataset]
    required_checks = [
        F.col(column["name"]).isNull()
        | ((F.col(column["name"]) == "") if column["type"] == "STRING" else F.lit(False))
        for column in spec["columns"] if column["required"]
    ]
    required_invalid = reduce(lambda left, check: left | check, required_checks, F.lit(False))
    key_window = Window.partitionBy(*spec["primary_key"]).orderBy(F.col("updated_at").desc_nulls_last(), F.col("source_row_id").desc())
    frame = (frame.withColumn("_duplicate_rank", F.row_number().over(key_window))
        .withColumn("_fk_invalid", F.lit(False)))
    for local_column, reference_dataset, reference_column in spec["foreign_keys"]:
        marker = f"_ref_{dataset}_{local_column}"
        reference = accepted[reference_dataset].select(F.col(reference_column).alias(marker)).distinct()
        frame = frame.join(F.broadcast(reference), frame[local_column] == reference[marker], "left")
        frame = frame.withColumn(
            "_fk_invalid",
            F.col("_fk_invalid") | (F.col(local_column).isNotNull() & F.col(marker).isNull()),
        ).drop(marker)
    fk_invalid = F.col("_fk_invalid")
    enum_invalid = reduce(
        lambda left, item: left | (F.col(item[0]).isNotNull() & ~F.col(item[0]).isin(item[1])),
        enum_rules.get(dataset, {}).items(),
        F.lit(False),
    )
    range_invalid = reduce(
        lambda left, name: left | (F.col(name).isNotNull() & (F.col(name) <= 0)),
        positive_rules.get(dataset, []),
        F.lit(False),
    )
    temporal_invalid = reduce(
        lambda left, pair: left | (
            F.col(pair[0]).isNotNull()
            & F.col(pair[1]).isNotNull()
            & (F.col(pair[1]) <= F.col(pair[0]))
        ),
        temporal_rules.get(dataset, []),
        F.lit(False),
    )
    scope_invalid = F.col("participant_key") != F.lit(participant_key)
    duplicate_invalid = F.col("_duplicate_rank") > 1
    quality_invalid = (
        required_invalid | F.col("_cast_invalid") | fk_invalid | enum_invalid
        | range_invalid | temporal_invalid | scope_invalid | duplicate_invalid
    )
    frame = frame.withColumn("_quality_invalid", quality_invalid)
    reason = F.concat_ws(",",
        F.when(required_invalid, F.lit("required_value")),
        F.when(F.col("_cast_invalid"), F.lit("invalid_type")),
        F.when(fk_invalid, F.lit("foreign_key")),
        F.when(enum_invalid, F.lit("invalid_enum")),
        F.when(range_invalid, F.lit("invalid_range")),
        F.when(temporal_invalid, F.lit("invalid_time_order")),
        F.when(scope_invalid, F.lit("participant_scope")),
        F.when(duplicate_invalid, F.lit("duplicate_key")),
    )
    source_columns = [column["name"] for column in spec["columns"]]
    issues = (frame.filter(F.col("_quality_invalid"))
        .withColumn("industry", F.lit(industry))
        .withColumn("dataset", F.lit(dataset))
        .withColumn("record_key", F.concat_ws("|", *[F.col(name).cast("string") for name in spec["primary_key"]]))
        .withColumn("reason_codes", reason)
        .withColumn("raw_payload_json", F.to_json(F.struct(*[F.col(name) for name in source_columns])))
        .withColumn("quarantined_at", F.current_timestamp())
        .select("participant_key", "industry", "dataset", "source_row_id", "record_key", "reason_codes", "raw_payload_json", "quarantined_at"))
    quality_frames.append(issues)
    clean = frame.filter(~F.col("_quality_invalid")).select(*source_columns)
    accepted[dataset] = clean
    bronze_count = frame.count()
    clean_count = clean.count()
    assert clean_count <= bronze_count, f"Silver count increased for {dataset}"
    clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("silver", dataset))
    print(f"Silver {dataset}: {clean_count} accepted rows")

quality = reduce(lambda left, right: left.unionByName(right), quality_frames)
quality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("silver", "quality_issues"))
quality_count = quality.count()
assert quality_count > 0, "The deterministic lab data must exercise the quarantine path"
print(f"Quality issues: {quality_count} rows")


## Expected result

Four clean Delta tables and a non-empty `quality_issues` table are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
